# NYISO Electricity Market Analysis with PySpark

**End-to-End Data Engineering & Machine Learning Pipeline**

This notebook demonstrates:
- PySpark data loading and schema definitions
- Spark SQL transformations and aggregations
- Window functions for time-series analysis
- Advanced EDA with interactive visualizations
- Feature engineering pipeline
- ML model training and evaluation

## 1. Setup and Imports

In [ ]:
# Import required libraries
import os
import sys
import glob

# Set JAVA_HOME for Spark
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"

# PySpark imports
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import *
from pyspark.sql.types import *

# ML imports
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import GBTRegressor, RandomForestRegressor, LinearRegression
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import RegressionEvaluator, BinaryClassificationEvaluator
from pyspark.ml.pipeline import Pipeline

# Visualization imports
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("All imports successful!")

In [ ]:
# Initialize Spark Session with optimized settings
spark = SparkSession.builder \
    .appName("NYISO_Analysis") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .config("spark.sql.session.timeZone", "America/New_York") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to reduce noise
spark.sparkContext.setLogLevel("WARN")

print(f"Spark Version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

In [ ]:
# Define paths
PROJECT_ROOT = os.path.dirname(os.getcwd())
LOAD_PATH = PROJECT_ROOT  # Load data in *pal_csv folders
PRICE_PATH = PROJECT_ROOT  # Price data in *realtime_zone_csv folders

print(f"Project Root: {PROJECT_ROOT}")

## 2. Schema Definitions

In [ ]:
# Define explicit schemas for better performance and type safety

# Price data schema (from realtime_zone_csv folders)
price_schema = StructType([
    StructField("Time Stamp", StringType(), True),
    StructField("Name", StringType(), True),
    StructField("PTID", IntegerType(), True),
    StructField("LBMP", DoubleType(), True),
    StructField("Marginal_Cost_Losses", DoubleType(), True),
    StructField("Marginal_Cost_Congestion", DoubleType(), True)
])

# Load data schema (from pal_csv folders)
load_schema = StructType([
    StructField("Time Stamp", StringType(), True),
    StructField("Time Zone", StringType(), True),
    StructField("Name", StringType(), True),
    StructField("PTID", IntegerType(), True),
    StructField("Load", DoubleType(), True)
])

print("Schemas defined successfully!")

## 3. Data Loading with PySpark

In [ ]:
# Function to find all CSV files matching a pattern
def find_csv_files(base_path, folder_pattern):
    """Find all CSV files in folders matching the pattern."""
    matching_dirs = glob.glob(os.path.join(base_path, folder_pattern))
    csv_files = []
    for d in matching_dirs:
        csv_files.extend(glob.glob(os.path.join(d, "*.csv")))
    return sorted(csv_files)

# Find price and load files
price_files = find_csv_files(PROJECT_ROOT, "*realtime_zone_csv")
load_files = find_csv_files(PROJECT_ROOT, "*pal_csv")

print(f"Found {len(price_files)} price data files")
print(f"Found {len(load_files)} load data files")
print(f"\nSample price file: {price_files[0] if price_files else 'None'}")
print(f"Sample load file: {load_files[0] if load_files else 'None'}")

In [ ]:
# Load Price Data with Spark
print("Loading price data...")

price_df = spark.read.csv(
    price_files,
    header=True,
    inferSchema=True,
    quote='"'
)

# Rename columns to be Spark-friendly
price_df = price_df \
    .withColumnRenamed("LBMP ($/MWHr)", "LBMP") \
    .withColumnRenamed("Marginal Cost Losses ($/MWHr)", "Marginal_Cost_Losses") \
    .withColumnRenamed("Marginal Cost Congestion ($/MWHr)", "Marginal_Cost_Congestion")

# Parse timestamp
price_df = price_df.withColumn(
    "timestamp",
    to_timestamp(col("Time Stamp"), "MM/dd/yyyy HH:mm:ss")
)

print(f"Price records loaded: {price_df.count():,}")
price_df.printSchema()

In [ ]:
# Load Load/Demand Data with Spark
print("Loading load data...")

load_df = spark.read.csv(
    load_files,
    header=True,
    inferSchema=True,
    quote='"'
)

# Parse timestamp
load_df = load_df.withColumn(
    "timestamp",
    to_timestamp(col("Time Stamp"), "MM/dd/yyyy HH:mm:ss")
)

print(f"Load records loaded: {load_df.count():,}")
load_df.printSchema()

In [ ]:
# Display sample data
print("=" * 60)
print("PRICE DATA SAMPLE")
print("=" * 60)
price_df.select("timestamp", "Name", "PTID", "LBMP", "Marginal_Cost_Losses", "Marginal_Cost_Congestion").show(10, truncate=False)

print("\n" + "=" * 60)
print("LOAD DATA SAMPLE")
print("=" * 60)
load_df.select("timestamp", "Name", "PTID", "Load", "Time Zone").show(10, truncate=False)

## 4. Spark SQL Data Exploration

In [ ]:
# Register DataFrames as SQL tables
price_df.createOrReplaceTempView("prices")
load_df.createOrReplaceTempView("loads")

print("Tables registered: prices, loads")

In [ ]:
# Basic statistics using Spark SQL
print("PRICE STATISTICS BY ZONE")
print("=" * 60)

spark.sql("""
    SELECT 
        Name as Zone,
        COUNT(*) as Records,
        ROUND(AVG(LBMP), 2) as Avg_Price,
        ROUND(MIN(LBMP), 2) as Min_Price,
        ROUND(MAX(LBMP), 2) as Max_Price,
        ROUND(STDDEV(LBMP), 2) as Std_Dev,
        ROUND(PERCENTILE(LBMP, 0.5), 2) as Median_Price
    FROM prices
    GROUP BY Name
    ORDER BY Avg_Price DESC
""").show(15, truncate=False)

In [ ]:
# Load statistics by zone
print("LOAD STATISTICS BY ZONE")
print("=" * 60)

spark.sql("""
    SELECT 
        Name as Zone,
        COUNT(*) as Records,
        ROUND(AVG(Load), 2) as Avg_Load_MW,
        ROUND(MIN(Load), 2) as Min_Load_MW,
        ROUND(MAX(Load), 2) as Max_Load_MW,
        ROUND(STDDEV(Load), 2) as Std_Dev
    FROM loads
    WHERE Load IS NOT NULL
    GROUP BY Name
    ORDER BY Avg_Load_MW DESC
""").show(15, truncate=False)

In [ ]:
# Time range analysis
print("DATA TIME RANGE")
print("=" * 60)

spark.sql("""
    SELECT 
        'Prices' as Dataset,
        MIN(timestamp) as Start_Date,
        MAX(timestamp) as End_Date,
        DATEDIFF(MAX(timestamp), MIN(timestamp)) as Days_Covered
    FROM prices
    WHERE timestamp IS NOT NULL
    
    UNION ALL
    
    SELECT 
        'Loads' as Dataset,
        MIN(timestamp) as Start_Date,
        MAX(timestamp) as End_Date,
        DATEDIFF(MAX(timestamp), MIN(timestamp)) as Days_Covered
    FROM loads
    WHERE timestamp IS NOT NULL
""").show(truncate=False)

## 5. Data Processing with Spark Transformations

In [ ]:
# Aggregate price data to hourly (from 5-minute intervals)
print("Aggregating price data to hourly...")

price_hourly = price_df \
    .withColumn("hour_timestamp", date_trunc("hour", col("timestamp"))) \
    .groupBy("hour_timestamp", "Name", "PTID") \
    .agg(
        avg("LBMP").alias("LBMP_avg"),
        min("LBMP").alias("LBMP_min"),
        max("LBMP").alias("LBMP_max"),
        stddev("LBMP").alias("LBMP_std"),
        avg("Marginal_Cost_Losses").alias("Marginal_Cost_Losses_avg"),
        avg("Marginal_Cost_Congestion").alias("Marginal_Cost_Congestion_avg"),
        count("*").alias("price_readings")
    )

print(f"Hourly price records: {price_hourly.count():,}")
price_hourly.show(5)

In [ ]:
# Process load data to hourly
print("Processing load data to hourly...")

load_hourly = load_df \
    .withColumn("hour_timestamp", date_trunc("hour", col("timestamp"))) \
    .groupBy("hour_timestamp", "Name", "PTID") \
    .agg(
        avg("Load").alias("Load_MW"),
        first("Time Zone").alias("Time_Zone")
    )

print(f"Hourly load records: {load_hourly.count():,}")
load_hourly.show(5)

In [ ]:
# Merge price and load data
print("Merging price and load data...")

merged_df = price_hourly.join(
    load_hourly,
    on=["hour_timestamp", "Name"],
    how="inner"
).drop(load_hourly["PTID"])

print(f"Merged records: {merged_df.count():,}")
merged_df.printSchema()

In [ ]:
# Add temporal features
print("Adding temporal features...")

import math

df_temporal = merged_df \
    .withColumn("hour", hour("hour_timestamp")) \
    .withColumn("day_of_week", dayofweek("hour_timestamp")) \
    .withColumn("day_of_month", dayofmonth("hour_timestamp")) \
    .withColumn("month", month("hour_timestamp")) \
    .withColumn("year", year("hour_timestamp")) \
    .withColumn("week_of_year", weekofyear("hour_timestamp")) \
    .withColumn("day_name", date_format("hour_timestamp", "EEEE")) \
    .withColumn("is_weekend", when(dayofweek("hour_timestamp").isin([1, 7]), 1).otherwise(0)) \
    .withColumn("is_peak_hour", when((hour("hour_timestamp") >= 7) & (hour("hour_timestamp") <= 22), 1).otherwise(0)) \
    .withColumn("hour_sin", sin(2 * math.pi * col("hour") / 24)) \
    .withColumn("hour_cos", cos(2 * math.pi * col("hour") / 24)) \
    .withColumn("month_sin", sin(2 * math.pi * col("month") / 12)) \
    .withColumn("month_cos", cos(2 * math.pi * col("month") / 12))

print(f"Records with temporal features: {df_temporal.count():,}")
df_temporal.select("hour_timestamp", "Name", "hour", "day_of_week", "month", "is_weekend", "is_peak_hour").show(5)

## 6. Window Functions for Time-Series Features

In [ ]:
# Define window specifications
window_24h = Window.partitionBy("Name").orderBy("hour_timestamp").rowsBetween(-24, -1)
window_168h = Window.partitionBy("Name").orderBy("hour_timestamp").rowsBetween(-168, -1)  # 1 week
window_lag = Window.partitionBy("Name").orderBy("hour_timestamp")

print("Window specifications defined")

In [ ]:
# Add rolling statistics using window functions
print("Computing rolling statistics with window functions...")

df_with_windows = df_temporal \
    .withColumn("price_ma_24h", avg("LBMP_avg").over(window_24h)) \
    .withColumn("price_std_24h", stddev("LBMP_avg").over(window_24h)) \
    .withColumn("price_min_24h", min("LBMP_avg").over(window_24h)) \
    .withColumn("price_max_24h", max("LBMP_avg").over(window_24h)) \
    .withColumn("price_ma_168h", avg("LBMP_avg").over(window_168h)) \
    .withColumn("load_ma_24h", avg("Load_MW").over(window_24h)) \
    .withColumn("load_std_24h", stddev("Load_MW").over(window_24h)) \
    .withColumn("load_min_24h", min("Load_MW").over(window_24h)) \
    .withColumn("load_max_24h", max("Load_MW").over(window_24h)) \
    .withColumn("load_ma_168h", avg("Load_MW").over(window_168h))

print("Rolling statistics added")

In [ ]:
# Add lag features
print("Adding lag features...")

df_with_lags = df_with_windows \
    .withColumn("price_lag_1h", lag("LBMP_avg", 1).over(window_lag)) \
    .withColumn("price_lag_2h", lag("LBMP_avg", 2).over(window_lag)) \
    .withColumn("price_lag_6h", lag("LBMP_avg", 6).over(window_lag)) \
    .withColumn("price_lag_12h", lag("LBMP_avg", 12).over(window_lag)) \
    .withColumn("price_lag_24h", lag("LBMP_avg", 24).over(window_lag)) \
    .withColumn("load_lag_1h", lag("Load_MW", 1).over(window_lag)) \
    .withColumn("load_lag_24h", lag("Load_MW", 24).over(window_lag)) \
    .withColumn("price_change_1h", col("LBMP_avg") - lag("LBMP_avg", 1).over(window_lag)) \
    .withColumn("load_change_1h", col("Load_MW") - lag("Load_MW", 1).over(window_lag))

print("Lag features added")

In [ ]:
# Add derived features
print("Adding derived features...")

df_features = df_with_lags \
    .withColumn("price_volatility", 
                when(col("price_ma_24h") != 0, col("price_std_24h") / abs(col("price_ma_24h"))).otherwise(0)) \
    .withColumn("load_ratio_24h",
                when(col("load_ma_24h") != 0, col("Load_MW") / col("load_ma_24h")).otherwise(1)) \
    .withColumn("price_deviation",
                when(col("price_std_24h") != 0, (col("LBMP_avg") - col("price_ma_24h")) / col("price_std_24h")).otherwise(0)) \
    .withColumn("congestion_impact",
                when(col("LBMP_avg") != 0, abs(col("Marginal_Cost_Congestion_avg")) / abs(col("LBMP_avg"))).otherwise(0)) \
    .withColumn("is_price_spike",
                when(col("LBMP_avg") > (col("price_ma_24h") + 3 * col("price_std_24h")), 1).otherwise(0))

# Drop rows with nulls from windowing
df_final = df_features.na.drop()

print(f"Final dataset records: {df_final.count():,}")
print(f"Total features: {len(df_final.columns)}")

In [ ]:
# Display all features
print("ALL FEATURES:")
print("=" * 60)
for i, col_name in enumerate(sorted(df_final.columns), 1):
    print(f"{i:2}. {col_name}")

## 7. Exploratory Data Analysis with Visualizations

In [ ]:
# Convert to Pandas for visualization
print("Converting to Pandas for visualization...")
df_pd = df_final.toPandas()
df_pd['date'] = pd.to_datetime(df_pd['hour_timestamp'])
print(f"Pandas DataFrame shape: {df_pd.shape}")

In [ ]:
# 1. Interactive 3D Load-Price-Time Visualization
print("Creating 3D visualization...")

# Sample for performance
sample_3d = df_pd.sample(n=min(5000, len(df_pd)), random_state=42)

fig = go.Figure(data=[go.Scatter3d(
    x=sample_3d['date'],
    y=sample_3d['Load_MW'],
    z=sample_3d['LBMP_avg'],
    mode='markers',
    marker=dict(
        size=4,
        color=sample_3d['LBMP_avg'],
        colorscale='Viridis',
        opacity=0.8,
        colorbar=dict(title="Price ($/MWh)")
    ),
    text=sample_3d['Name']
)])

fig.update_layout(
    title='3D Visualization: Load vs Price Over Time',
    scene=dict(
        xaxis_title='Time',
        yaxis_title='Load (MW)',
        zaxis_title='Price ($/MWh)'
    ),
    width=900,
    height=700
)
fig.show()

In [ ]:
# 2. Heatmaps: Load and Price by Hour and Month
fig, axes = plt.subplots(2, 1, figsize=(14, 12))

# Load Heatmap
load_pivot = df_pd.pivot_table(
    values='Load_MW',
    index='hour',
    columns='month',
    aggfunc='mean'
)

sns.heatmap(load_pivot, cmap='YlOrRd', ax=axes[0], annot=True, fmt='.0f', cbar_kws={'label': 'MW'})
axes[0].set_title('Average Load by Hour and Month', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Hour of Day')

# Price Heatmap
price_pivot = df_pd.pivot_table(
    values='LBMP_avg',
    index='hour',
    columns='month',
    aggfunc='mean'
)

sns.heatmap(price_pivot, cmap='coolwarm', ax=axes[1], annot=True, fmt='.1f', cbar_kws={'label': '$/MWh'})
axes[1].set_title('Average Price by Hour and Month', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Hour of Day')

plt.tight_layout()
plt.show()

In [ ]:
# 3. Interactive Pattern Analysis
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Load Distribution by Day of Week',
        'Price Distribution by Day of Week',
        'Load-Price Correlation by Month',
        'Congestion vs Price'
    )
)

# Load by day
fig.add_trace(
    go.Box(x=df_pd['day_name'], y=df_pd['Load_MW'], name='Load', marker_color='#00d4ff'),
    row=1, col=1
)

# Price by day
fig.add_trace(
    go.Box(x=df_pd['day_name'], y=df_pd['LBMP_avg'], name='Price', marker_color='#ff6b6b'),
    row=1, col=2
)

# Monthly correlation
monthly_corr = df_pd.groupby('month').apply(
    lambda x: x['Load_MW'].corr(x['LBMP_avg']), include_groups=False
).reset_index()
monthly_corr.columns = ['month', 'correlation']

fig.add_trace(
    go.Scatter(x=monthly_corr['month'], y=monthly_corr['correlation'], 
               mode='lines+markers', name='Correlation', marker_color='#4ecdc4'),
    row=2, col=1
)

# Congestion vs Price (sampled)
sample = df_pd.sample(n=min(2000, len(df_pd)), random_state=42)
fig.add_trace(
    go.Scatter(
        x=sample['Marginal_Cost_Congestion_avg'],
        y=sample['LBMP_avg'],
        mode='markers',
        marker=dict(color=sample['Load_MW'], colorscale='Viridis', size=5, opacity=0.6),
        name='Congestion Impact'
    ),
    row=2, col=2
)

fig.update_layout(height=800, width=1100, title_text="Advanced Pattern Analysis", showlegend=False)
fig.show()

In [ ]:
# 4. Price and Load Time Series by Zone
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Daily Average Price by Zone', 'Daily Average Load by Zone'))

# Aggregate to daily
daily_zone = df_pd.groupby([df_pd['date'].dt.date, 'Name']).agg({
    'LBMP_avg': 'mean',
    'Load_MW': 'mean'
}).reset_index()
daily_zone.columns = ['date', 'Name', 'LBMP_avg', 'Load_MW']

colors = px.colors.qualitative.Set2

for i, zone in enumerate(daily_zone['Name'].unique()[:6]):  # Top 6 zones
    zone_data = daily_zone[daily_zone['Name'] == zone]
    fig.add_trace(
        go.Scatter(x=zone_data['date'], y=zone_data['LBMP_avg'], name=zone, 
                   line=dict(color=colors[i % len(colors)])),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=zone_data['date'], y=zone_data['Load_MW'], name=zone, 
                   line=dict(color=colors[i % len(colors)]), showlegend=False),
        row=2, col=1
    )

fig.update_layout(height=700, width=1100, title_text="Price & Load Time Series by Zone")
fig.update_yaxes(title_text="Price ($/MWh)", row=1, col=1)
fig.update_yaxes(title_text="Load (MW)", row=2, col=1)
fig.show()

In [ ]:
# 5. Volatility Analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Price volatility distribution
axes[0].hist(df_pd['price_volatility'].clip(0, 2), bins=50, color='#00d4ff', edgecolor='black', alpha=0.7)
axes[0].set_title('Price Volatility Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Volatility (Std/Mean)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df_pd['price_volatility'].median(), color='red', linestyle='--', label=f"Median: {df_pd['price_volatility'].median():.2f}")
axes[0].legend()

# Volatility by hour
hourly_vol = df_pd.groupby('hour')['price_volatility'].mean()
axes[1].bar(hourly_vol.index, hourly_vol.values, color='#ff6b6b', edgecolor='black', alpha=0.7)
axes[1].set_title('Average Price Volatility by Hour', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Avg Volatility')
axes[1].set_xticks(range(0, 24, 2))

plt.tight_layout()
plt.show()

In [ ]:
# 6. Correlation Matrix
numeric_cols = ['LBMP_avg', 'Load_MW', 'Marginal_Cost_Losses_avg', 'Marginal_Cost_Congestion_avg',
                'price_ma_24h', 'load_ma_24h', 'price_volatility', 'hour', 'month', 'is_weekend']

corr_matrix = df_pd[numeric_cols].corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Price Spike Analysis

In [ ]:
# Analyze price spikes
spike_analysis = df_final.groupBy("is_price_spike").agg(
    count("*").alias("count"),
    avg("LBMP_avg").alias("avg_price"),
    avg("Load_MW").alias("avg_load"),
    avg("Marginal_Cost_Congestion_avg").alias("avg_congestion")
)

print("PRICE SPIKE ANALYSIS")
print("=" * 60)
spike_analysis.show()

# Spike rate
total = df_final.count()
spikes = df_final.filter(col("is_price_spike") == 1).count()
print(f"\nSpike Rate: {spikes/total*100:.2f}% ({spikes:,} of {total:,} records)")

In [ ]:
# Spike distribution by hour
spike_by_hour = df_pd[df_pd['is_price_spike'] == 1].groupby('hour').size()

plt.figure(figsize=(12, 5))
plt.bar(spike_by_hour.index, spike_by_hour.values, color='#ff6b6b', edgecolor='black')
plt.title('Price Spikes by Hour of Day', fontsize=14, fontweight='bold')
plt.xlabel('Hour')
plt.ylabel('Number of Spikes')
plt.xticks(range(24))
plt.grid(axis='y', alpha=0.3)
plt.show()

## 9. Machine Learning: Price Prediction Model

In [ ]:
# Prepare features for ML
feature_cols = [
    'hour', 'day_of_week', 'month', 'is_weekend', 'is_peak_hour',
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos',
    'Load_MW', 'load_ma_24h', 'load_std_24h', 'load_ratio_24h',
    'price_lag_1h', 'price_lag_24h', 'price_ma_24h', 'price_std_24h',
    'Marginal_Cost_Losses_avg', 'Marginal_Cost_Congestion_avg',
    'price_volatility', 'congestion_impact'
]

print(f"Using {len(feature_cols)} features for prediction")

In [ ]:
# Build ML Pipeline
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features_raw",
    handleInvalid="skip"
)

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withStd=True,
    withMean=True
)

gbt = GBTRegressor(
    labelCol="LBMP_avg",
    featuresCol="features",
    predictionCol="predicted_price",
    maxDepth=8,
    maxIter=50,
    seed=42
)

pipeline = Pipeline(stages=[assembler, scaler, gbt])

print("ML Pipeline created")

In [ ]:
# Split data
train_df, test_df = df_final.randomSplit([0.8, 0.2], seed=42)
train_df.cache()
test_df.cache()

print(f"Training samples: {train_df.count():,}")
print(f"Test samples: {test_df.count():,}")

In [ ]:
# Train model
print("Training GBT model...")
model = pipeline.fit(train_df)
print("Model trained!")

In [ ]:
# Make predictions
predictions = model.transform(test_df)

# Evaluate
evaluator_rmse = RegressionEvaluator(labelCol="LBMP_avg", predictionCol="predicted_price", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="LBMP_avg", predictionCol="predicted_price", metricName="r2")
evaluator_mae = RegressionEvaluator(labelCol="LBMP_avg", predictionCol="predicted_price", metricName="mae")

rmse = evaluator_rmse.evaluate(predictions)
r2 = evaluator_r2.evaluate(predictions)
mae = evaluator_mae.evaluate(predictions)

print("=" * 60)
print("PRICE PREDICTION MODEL RESULTS")
print("=" * 60)
print(f"R² Score:  {r2:.4f}")
print(f"RMSE:      {rmse:.2f} $/MWh")
print(f"MAE:       {mae:.2f} $/MWh")

In [ ]:
# Feature Importance
gbt_model = model.stages[-1]
importances = gbt_model.featureImportances.toArray()

feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': importances
}).sort_values('importance', ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(feature_importance['feature'], feature_importance['importance'], color='#00d4ff')
plt.xlabel('Importance')
plt.title('Feature Importance for Price Prediction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Prediction vs Actual Plot
pred_pd = predictions.select("LBMP_avg", "predicted_price").toPandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
axes[0].scatter(pred_pd['LBMP_avg'], pred_pd['predicted_price'], alpha=0.3, s=10, c='#00d4ff')
axes[0].plot([pred_pd['LBMP_avg'].min(), pred_pd['LBMP_avg'].max()], 
             [pred_pd['LBMP_avg'].min(), pred_pd['LBMP_avg'].max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Price ($/MWh)')
axes[0].set_ylabel('Predicted Price ($/MWh)')
axes[0].set_title(f'Predicted vs Actual (R² = {r2:.3f})', fontsize=12, fontweight='bold')

# Residual distribution
residuals = pred_pd['LBMP_avg'] - pred_pd['predicted_price']
axes[1].hist(residuals, bins=50, color='#ff6b6b', edgecolor='black', alpha=0.7)
axes[1].axvline(0, color='black', linestyle='--', lw=2)
axes[1].set_xlabel('Residual ($/MWh)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Residual Distribution', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 10. Save Results

In [ ]:
# Save processed data to parquet
output_path = os.path.join(PROJECT_ROOT, "data", "processed", "spark_featured_data")
df_final.write.mode("overwrite").parquet(output_path)
print(f"Processed data saved to: {output_path}")

In [ ]:
# Save model
model_path = os.path.join(PROJECT_ROOT, "models", "spark_price_predictor")
model.write().overwrite().save(model_path)
print(f"Model saved to: {model_path}")

In [ ]:
# Cleanup
train_df.unpersist()
test_df.unpersist()
spark.stop()
print("\nSpark session stopped. Analysis complete!")